# Berlin Wind Power Multi-Class Classification
Federated Gradient Boosting implementation from scratch with separated functions and periodic evaluation.

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [4]:
df = pd.read_excel('Berlin_wind_multiclass1.xlsx')

power = df['X50Hertz..MW.']
boundaries = power.quantile([0.2, 0.4, 0.6, 0.8]).values
conditions = [
    power <= boundaries[0],
    (power > boundaries[0]) & (power <= boundaries[1]),
    (power > boundaries[1]) & (power <= boundaries[2]),
    (power > boundaries[2]) & (power <= boundaries[3]),
    power > boundaries[3]
]
df['target'] = np.select(conditions, [0, 1, 2, 3, 4])

feature_cols = [col for col in df.columns if col not in ['X50Hertz..MW.', 'target', 'Year', 'Minute']]
X = df[feature_cols].values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
MAX_DEPTH = 5
NUM_TREES = 300
LEARNING_RATE = 0.1
NUM_CLASSES = 5
L2_REG = 1.0
GAMMA = 0.0
NUM_CLIENTS = 3

In [6]:
def compute_gradients_hessians(preds, labels, num_classes=NUM_CLASSES):
    exp_preds = np.exp(preds - np.max(preds, axis=1, keepdims=True))
    probs = exp_preds / np.sum(exp_preds, axis=1, keepdims=True)
    y_onehot = np.eye(num_classes)[labels]
    grad = probs - y_onehot
    hess = probs * (1.0 - probs)
    return grad, hess, probs

In [7]:
def calculate_split_gain(sum_g_left, sum_h_left, sum_g_right, sum_h_right, sum_g_all, sum_h_all):
    def score(g, h):
        return np.sum(g ** 2) / (np.sum(h) + L2_REG)
    score_left = score(sum_g_left, sum_h_left)
    score_right = score(sum_g_right, sum_h_right)
    score_all = score(sum_g_all, sum_h_all)
    gain = 0.5 * (score_left + score_right - score_all) - GAMMA
    return gain

In [8]:
# def build_scratch_tree(X_data, grads, hessians, current_depth=0):
#     n_samples, n_features = X_data.shape
#     if current_depth >= MAX_DEPTH or n_samples < 2:
#         leaf_weight = -np.sum(grads, axis=0) / (np.sum(hessians, axis=0) + L2_REG)
#         return {'is_leaf': True, 'weight': leaf_weight}
    
#     best_gain = -float('inf')
#     best_split = None
#     sum_g_all = np.sum(grads, axis=0)
#     sum_h_all = np.sum(hessians, axis=0)
    
#     for feat_idx in range(n_features):
#         feature_vals = X_data[:, feat_idx]
#         sorted_indices = np.argsort(feature_vals)
#         sorted_vals = feature_vals[sorted_indices]
#         sorted_g = grads[sorted_indices]
#         sorted_h = hessians[sorted_indices]
        
#         for i in range(1, n_samples):
#             if sorted_vals[i] == sorted_vals[i - 1]:
#                 continue
#             sum_g_left = np.sum(sorted_g[:i], axis=0)
#             sum_h_left = np.sum(sorted_h[:i], axis=0)
#             sum_g_right = sum_g_all - sum_g_left
#             sum_h_right = sum_h_all - sum_h_left
#             gain = calculate_split_gain(sum_g_left, sum_h_left, sum_g_right, sum_h_right, sum_g_all, sum_h_all)
#             if gain > best_gain:
#                 best_gain = gain
#                 threshold = (sorted_vals[i] + sorted_vals[i - 1]) / 2.0
#                 best_split = {
#                     'feature_idx': feat_idx,
#                     'threshold': threshold,
#                     'left_indices': sorted_indices[:i],
#                     'right_indices': sorted_indices[i:]
#                 }
                
#     if best_gain <= 0 or best_split is None:
#         leaf_weight = -sum_g_all / (sum_h_all + L2_REG)
#         return {'is_leaf': True, 'weight': leaf_weight}
        
#     left_tree = build_scratch_tree(X_data[best_split['left_indices']], grads[best_split['left_indices']], hessians[best_split['left_indices']], current_depth + 1)
#     right_tree = build_scratch_tree(X_data[best_split['right_indices']], grads[best_split['right_indices']], hessians[best_split['right_indices']], current_depth + 1)
    
#     return {
#         'is_leaf': False,
#         'feature_idx': best_split['feature_idx'],
#         'threshold': best_split['threshold'],
#         'left': left_tree,
#         'right': right_tree
#     }

In [9]:
def build_scratch_tree(X_data, grads, hessians, current_depth=0):
    n_samples, n_features = X_data.shape

    sum_g_all = np.sum(grads, axis=0)
    sum_h_all = np.sum(hessians, axis=0)

    if current_depth >= MAX_DEPTH or n_samples < 2:
        leaf_weight = -sum_g_all / (sum_h_all + L2_REG)
        return {'is_leaf': True, 'weight': leaf_weight}

    best_gain = -float('inf')
    best_split = None

    for feat_idx in range(n_features):
        feature_vals = X_data[:, feat_idx]
        sorted_indices = np.argsort(feature_vals)

        sorted_vals = feature_vals[sorted_indices]
        sorted_g = grads[sorted_indices]
        sorted_h = hessians[sorted_indices]

        cum_g = np.cumsum(sorted_g, axis=0)
        cum_h = np.cumsum(sorted_h, axis=0)

        valid = sorted_vals[:-1] < sorted_vals[1:]

        if not np.any(valid):
            continue

        left_g = cum_g[:-1][valid]
        left_h = cum_h[:-1][valid]

        right_g = sum_g_all - left_g
        right_h = sum_h_all - left_h

        gain = 0.5 * (
            np.sum(left_g**2, axis=1) / (np.sum(left_h, axis=1) + L2_REG)
            + np.sum(right_g**2, axis=1) / (np.sum(right_h, axis=1) + L2_REG)
            - np.sum(sum_g_all**2) / (np.sum(sum_h_all) + L2_REG)
        ) - GAMMA

        best_idx = np.argmax(gain)

        if gain[best_idx] > best_gain:
            best_gain = gain[best_idx]

            split_positions = np.flatnonzero(valid)
            split_pos = split_positions[best_idx] + 1

            best_split = {
                'feature_idx': feat_idx,
                'threshold': (
                    sorted_vals[split_pos] +
                    sorted_vals[split_pos - 1]
                ) / 2.0,
                'left_indices': sorted_indices[:split_pos],
                'right_indices': sorted_indices[split_pos:]
            }

    if best_gain <= 0 or best_split is None:
        leaf_weight = -sum_g_all / (sum_h_all + L2_REG)
        return {'is_leaf': True, 'weight': leaf_weight}

    left_tree = build_scratch_tree(
        X_data[best_split['left_indices']],
        grads[best_split['left_indices']],
        hessians[best_split['left_indices']],
        current_depth + 1
    )

    right_tree = build_scratch_tree(
        X_data[best_split['right_indices']],
        grads[best_split['right_indices']],
        hessians[best_split['right_indices']],
        current_depth + 1
    )

    return {
        'is_leaf': False,
        'feature_idx': best_split['feature_idx'],
        'threshold': best_split['threshold'],
        'left': left_tree,
        'right': right_tree
    }

In [10]:
def predict_tree(tree, x):
    if tree['is_leaf']:
        return tree['weight']
    if x[tree['feature_idx']] <= tree['threshold']:
        return predict_tree(tree['left'], x)
    else:
        return predict_tree(tree['right'], x)

In [11]:
def predict_ensemble(trees, X_data):
    preds = np.zeros((len(X_data), NUM_CLASSES))
    for x_idx, x in enumerate(X_data):
        for tree in trees:
            preds[x_idx] += LEARNING_RATE * predict_tree(tree, x)
    return preds

In [12]:
client_chunks = np.array_split(range(len(X_train)), NUM_CLIENTS)
train_preds = np.zeros((len(X_train), NUM_CLASSES))
trees = []

for tree_idx in range(NUM_TREES):
    grad_all, hess_all, _ = compute_gradients_hessians(train_preds, y_train, NUM_CLASSES)
    client_gradients = []
    client_hessians = []
    
    for client_id, indices in enumerate(client_chunks):
        client_gradients.append(grad_all[indices])
        client_hessians.append(hess_all[indices])
        
    aggregated_grads = np.concatenate(client_gradients, axis=0)
    aggregated_hess = np.concatenate(client_hessians, axis=0)
    
    tree = build_scratch_tree(X_train, aggregated_grads, aggregated_hess, current_depth=0)
    trees.append(tree)
    
    for x_idx, x in enumerate(X_train):
        train_preds[x_idx] += LEARNING_RATE * predict_tree(tree, x)
        
    if (tree_idx + 1) % 10 == 0 or tree_idx == NUM_TREES - 1:
        current_test_preds = predict_ensemble(trees, X_test)
        test_probs = np.exp(current_test_preds - np.max(current_test_preds, axis=1, keepdims=True))
        test_probs /= np.sum(test_probs, axis=1, keepdims=True)
        current_test_labels = np.argmax(test_probs, axis=1)
        acc = accuracy_score(y_test, current_test_labels)
        print(f"Round {tree_idx + 1:3d}/{NUM_TREES} | Test Accuracy: {acc * 100:.2f}%")

final_test_preds = predict_ensemble(trees, X_test)
final_test_probs = np.exp(final_test_preds - np.max(final_test_preds, axis=1, keepdims=True))
final_test_probs /= np.sum(final_test_probs, axis=1, keepdims=True)
final_test_labels = np.argmax(final_test_probs, axis=1)
final_accuracy = accuracy_score(y_test, final_test_labels)
print(f"\nFinal Test Accuracy: {final_accuracy * 100:.2f}%")

Round  10/300 | Test Accuracy: 61.32%
Round  20/300 | Test Accuracy: 65.22%
Round  30/300 | Test Accuracy: 67.79%
Round  40/300 | Test Accuracy: 69.68%
Round  50/300 | Test Accuracy: 71.60%
Round  60/300 | Test Accuracy: 72.59%
Round  70/300 | Test Accuracy: 73.92%
Round  80/300 | Test Accuracy: 74.74%
Round  90/300 | Test Accuracy: 75.90%
Round 100/300 | Test Accuracy: 76.84%
Round 110/300 | Test Accuracy: 77.33%
Round 120/300 | Test Accuracy: 77.90%
Round 130/300 | Test Accuracy: 78.75%
Round 140/300 | Test Accuracy: 79.13%
Round 150/300 | Test Accuracy: 79.84%
Round 160/300 | Test Accuracy: 80.42%
Round 170/300 | Test Accuracy: 81.01%
Round 180/300 | Test Accuracy: 81.31%
Round 190/300 | Test Accuracy: 81.69%


KeyboardInterrupt: 